# Cell-model ensemble — training notebook
Tests the core question cheaply: **do frozen foundation-model gene embeddings (Geneformer [+scGPT]) add anything OVER our integrated biological features for essentiality?**
Baseline to beat = our features-only MLP **AUC 0.973** (graph GNN/GAT already *lost*: 0.947/0.943 -> combiner = MLP).

**All data + artifacts are saved to Google Drive** (`MyDrive/cell_model/`). Small results are also written for committing back to the repo.

### GPU + time
| phase | needs | GPU | time |
|---|---|---|---|
| **1. FROZEN gene embeddings + combiner** (this notebook) | grab Geneformer gene-embedding matrix + train tiny MLP. **No fine-tuning.** | **CPU or free T4** | ~5-10 min |
| 2. FROZEN cell embeddings (Tabula Sapiens ~500k cells) | run Geneformer **forward only** (no training) - GPU is for throughput, not gradients | **T4** subset(50k)~30 min / **A100** full~45 min | hours on T4 |
| 3. LoRA fine-tune - **OPTIONAL FALLBACK, only if the frozen ensemble ties/loses** | actually adapts weights | **A100 / L4** | hours |

**Strategy: frozen embeddings, not fine-tuning.** Phases 1 and 2 never change the model weights - they extract embeddings (forward pass) and train a tiny combiner. Phase 3 (real fine-tuning) is a fallback we do ONLY if the frozen ensemble fails to beat the 0.973 baseline. Do Phase 1 on **free T4**; only pay for **A100** for the Phase-2 atlas scan. Set *Runtime -> T4 GPU*.


In [ ]:
# [1] Mount Google Drive - everything persists here
from google.colab import drive; drive.mount('/content/drive')
import os; PROJ='/content/drive/MyDrive/cell_model'; os.makedirs(PROJ, exist_ok=True)
print('all data/artifacts ->', PROJ)


In [ ]:
# [2] install
!pip -q install torch scikit-learn pandas numpy transformers huggingface_hub


In [ ]:
# [3] OUR priors + labels (from the repo) -> save copy to Drive
import pandas as pd, numpy as np
df=pd.read_csv('https://raw.githubusercontent.com/nikku03/cell/claude/vectorize-gex-propensity-NRqBW/outputs/orphan/integrated_cell_human.csv')
df.to_csv(f'{PROJ}/integrated_cell_human.csv', index=False)
FEATS=['loeuf','is_tf','regulon_out','regulators_in','ppi_degree','n_pathways','cpg_promoter','enhancers','n_diseases']
for c in ['regulon_out','regulators_in','ppi_degree','n_pathways','enhancers','n_diseases']: df[c]=np.log1p(df[c].fillna(0))
df['loeuf']=df['loeuf'].fillna(df['loeuf'].median()); df[FEATS]=df[FEATS].fillna(0)
y=df['essential']; mask=y.notna().values
print('labeled:', int(mask.sum()), '| essential:', int((y==1).sum()))


In [ ]:
# [4] (optional, recommended) DepMap measured essentiality -> save to Drive
# Get the CURRENT CRISPRGeneEffect file id from https://depmap.org/portal/download (it changes per release).
# !wget -q -O {PROJ}/CRISPRGeneEffect.csv 'https://ndownloader.figshare.com/files/<CURRENT_ID>'
# de=pd.read_csv(f'{PROJ}/CRISPRGeneEffect.csv', index_col=0)
# de.columns=[c.split(' ')[0] for c in de.columns]
# ce=(de.median(0) < -0.5).astype(int)          # common-essential per gene (measured)
# df['depmap_ess']=df['gene'].map(ce); y=df['depmap_ess']; mask=y.notna().values
print('DepMap optional: uncomment to upgrade labels from Hart CEG/NEG to measured essentiality')


In [ ]:
# [5] FROZEN Geneformer gene embeddings -> save npz to Drive (small, also committable)
import pickle, numpy as np, torch
from huggingface_hub import hf_hub_download
from transformers import AutoModel
def geneformer_gene_emb(symbols):
    n2i=pickle.load(open(hf_hub_download('ctheodoris/Geneformer','geneformer/gene_dictionaries_30m/gene_name_id_dict_gc30M.pkl'),'rb'))
    i2t=pickle.load(open(hf_hub_download('ctheodoris/Geneformer','geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl'),'rb'))
    W=AutoModel.from_pretrained('ctheodoris/Geneformer').get_input_embeddings().weight.detach().cpu().numpy()
    D=W.shape[1]; M=np.zeros((len(symbols),D),np.float32); have=np.zeros(len(symbols),bool)
    for k,s in enumerate(symbols):
        eid=n2i.get(s); t=i2t.get(eid) if eid else None
        if t is not None and t<len(W): M[k]=W[t]; have[k]=True
    return M, have
EMB, HAVE = geneformer_gene_emb(df['gene'].tolist())
np.savez_compressed(f'{PROJ}/geneformer_gene_emb.npz', emb=EMB, have=HAVE, genes=df['gene'].values)
print('Geneformer emb:', EMB.shape, '| genes covered:', int(HAVE.sum()), '-> saved to Drive')


In [ ]:
# [6] HONEST benchmark: our features  vs  our features + Geneformer embedding
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_predict; from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler; import json
def bench(extra=None):
    X=df.loc[mask,FEATS].values.astype(float)
    if extra is not None: X=np.hstack([X, extra[mask]])
    X=StandardScaler().fit_transform(X); yy=y[mask].astype(int).values
    p=cross_val_predict(MLPClassifier(hidden_layer_sizes=(64,),max_iter=300,alpha=1e-3),X,yy,cv=5,method='predict_proba')[:,1]
    return roc_auc_score(yy,p)
base=bench(None); combo=bench(EMB)
res={'baseline_features_only':round(float(base),3),'features_plus_geneformer':round(float(combo),3),
     'verdict':'ensemble helps' if combo>base+0.003 else 'ties/loses -> keep interpretable map'}
json.dump(res, open(f'{PROJ}/phase1_result.json','w'), indent=2); print(res)
json.dump(res, open('phase1_result.json','w'), indent=2)  # small copy to download + commit


### Committing small artifacts back to the repo
`phase1_result.json` and `geneformer_gene_emb.npz` are small. Download them from Drive and commit to `outputs/orphan/`, or (if you set a GitHub PAT) `git clone` the repo in a cell and push. **Do not paste your PAT into a shared notebook.** Large files (cell embeddings, atlas) stay in Drive only.


## Phase 2 - cell-type state (the keystone), the CHEAP honest way
Phase 1 proved foundation *gene embeddings* add nothing for essentiality (0.53 alone), so we do **not** use them here. The cell-type layer we lack is simply **measured expression per cell type** - Tabula Sapiens gives it directly. **This is CPU/RAM-bound, not GPU** (the A100 sits idle; a **high-RAM** runtime helps more). Result = per-cell-type active gene sets + cell-type-specific networks.


In [ ]:
# [7] load a Tabula Sapiens subset from the CELLxGENE census
!pip -q install cellxgene-census scanpy
import cellxgene_census, scanpy as sc, numpy as np, pandas as pd
TISSUES=['liver','pancreas','heart','kidney']   # start small; add 'blood','lung' if RAM allows
with cellxgene_census.open_soma(census_version='stable') as census:
    adata=cellxgene_census.get_anndata(census,'Homo sapiens',
        obs_value_filter=f"tissue_general in {TISSUES} and is_primary_data==True",
        column_names={'obs':['cell_type','tissue_general'],'var':['feature_name']})
adata.var_names=adata.var['feature_name'].astype(str)
adata.write(f'{PROJ}/tabula_subset.h5ad')
print(adata.shape,'|',int(adata.obs.cell_type.nunique()),'cell types')


In [ ]:
# [8] per-cell-type mean expression -> active gene set per cell type -> save to Drive
sc.pp.normalize_total(adata,target_sum=1e4); sc.pp.log1p(adata)
cts=[c for c in adata.obs.cell_type.value_counts().index if (adata.obs.cell_type==c).sum()>=50]
expr=pd.DataFrame(index=adata.var_names)
for c in cts: expr[c]=np.asarray(adata[adata.obs.cell_type==c].X.mean(0)).ravel()
expr.to_csv(f'{PROJ}/celltype_expression.csv')
active={c:set(expr.index[expr[c]>0.5]) for c in cts}
print(len(cts),'cell types | mean active genes/type:',int(np.mean([len(a) for a in active.values()])))


In [ ]:
# [9] integrate with OUR map: cell-type-specific network + master-TF validation
reg=pd.read_csv('https://raw.githubusercontent.com/nikku03/cell/claude/vectorize-gex-propensity-NRqBW/data/external_data/human/collectri.tsv',sep='\t')
edges=list(zip(reg['source_genesymbol'],reg['target_genesymbol']))
MASTERS={'hepatocyte':['HNF4A','HNF1A','FOXA2'],'cardiac muscle cell':['NKX2-5','GATA4','TBX5'],
 'pancreatic beta cell':['PDX1','NKX6-1'],'natural killer cell':['EOMES','TBX21'],
 'macrophage':['SPI1','CEBPB'],'endothelial cell':['ERG','FLI1'],'kidney epithelial cell':['PAX8','HNF1B']}
import json; rep={}
for c in cts:
    A=active[c]; net=sum(1 for a,b in edges if a in A and b in A)
    rep[c]={'active_genes':len(A),'celltype_network_edges':net,'master_TFs_active':[m for m in MASTERS.get(c,[]) if m in A]}
print('cell-type-specific active networks (should differ per type; masters should light up correctly):')
for c in MASTERS:
    if c in rep: print(f"  {c:24s} edges={rep[c]['celltype_network_edges']:6d} masters_on={rep[c]['master_TFs_active']}")
json.dump(rep, open(f'{PROJ}/celltype_layer.json','w'), indent=2)
json.dump(rep, open('celltype_layer.json','w'), indent=2)  # small copy to download + commit
print('saved per-cell-type active networks -> Drive (celltype_layer.json)')
